In [ ]:
"""
exp12 Step 01: VIX Fear Re-entry Exploration (EXPLORATION 2009-08-13 -> 2014-12-31 ONLY; context, PROTOCOL.md §6)

The 6 candidates of PROTOCOL.md §4 (exits E1 = exp04's x = 0, n = 1 and E2 = exp07's k = 4; modifications M1 fear-fade
re-entry, M2 term-structure re-entry, M3 no exit into fear; _prev VIX timing only) are replayed as ONE continuous path
over 2009-08-13 -> 2014-12-31 (start invested, no forced buy-back; the VIX / VIX3M term feature exists from 2009-08-13),
against buy-and-hold and the unmodified E1 and E2 over the same sessions. SPY bars and every VIX load are cut at
so.config.EXPLORATION_DATA_CUTOFF_DATE_STR (2015-03-18): nothing of the walk-forward period is read.

Context only: the result changes nothing in the protocol. Caveat: 2010-2011 and their VIX peaks are known to everyone.
Outputs: store04_experiments/exp12_vix_fear_reentry/step01_exploration_data/. Trials: 6 (stage "exploration").
"""
None

In [ ]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp12_vix_fear_reentry import config as exp_config
# IMPORT DATA LOADING, EXECUTION, DAILY FEATURE AND VIX FEATURE FUNCTIONS
from so.core.raw_data import get_complete_ohlcv_pdf
from so.core.trade_execution import get_ohlcv_array_dict
from so.features.daily_features import get_daily_feature_pdf
from so.features.vix_features import get_vix_daily_feature_dict
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import write_csv_file_to_path
# IMPORT EXIT AND RE-ENTRY SIMULATION FUNCTIONS
from so.core.reentry_simulation import get_buy_and_hold_result_dict
# IMPORT THE CONTINUOUS REPLAY FUNCTIONS
from so.core.continuous_replay import simulate_continuous_replay_dict, get_episode_scorecard_pdf, get_replay_summary_dict
# IMPORT THE EXPERIMENT FUNCTIONS
from experiments.exp12_vix_fear_reentry.rules import get_rule_candidate_list, get_unmodified_candidate_list, get_rule_builder_func, get_vix_scorecard_pdf
# IMPORT TRIAL LOG FUNCTIONS
from so.core.trial_log import get_config_hash_str, log_trial_pdf, read_trial_log_pdf, get_trial_count_pdf, print_previous_trial_warning

In [ ]:
# CHECK THE REAL DATA ONLY OUTSIDE THE SYNTHETIC SMOKE RUN (scripts/smoke_notebook.py)
CHECK_REAL_DATA_BOOL = __name__ != "__smoke__"
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Experiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step01_exploration_data")
print(f"Output folder:\t{output_path_str}")
# CALL FUNCTION TO WARN ABOUT EARLIER ENTRIES OF THIS STAGE (A RERUN ADDS TRIALS)
previous_trial_count = print_previous_trial_warning(exp_config, "exploration")

In [ ]:
# CALL FUNCTION TO READ THE MINUTE BARS UP TO THE EXPLORATION DATA CUTOFF (NOTHING FROM THE WALK-FORWARD PERIOD IS READ)
cutoff_date_str = exp_config.EXPLORATION_DATA_CUTOFF_DATE_STR
complete_ohlcv_pdf = get_complete_ohlcv_pdf(cutoff_date_str_in=cutoff_date_str)
assert complete_ohlcv_pdf["date"].max() <= pd.Timestamp(cutoff_date_str).date()
# CALL FUNCTION TO CONVERT THE DATA TO ARRAYS AND BUILD THE DAILY TABLE
ohlcv_array_dict = get_ohlcv_array_dict(complete_ohlcv_pdf)
daily_pdf = get_daily_feature_pdf(ohlcv_array_dict)
# CALL FUNCTION TO LOAD THE VIX AND VIX3M DAILY BARS WITH THE SAME CUTOFF AND BUILD THE _prev FEATURES
vix_dict = get_vix_daily_feature_dict(daily_pdf, cutoff_date_str, [exp_config.VIX_VARIANT_STR])
vix_feature_pdf = vix_dict["feature_pdf"]
assert all(last_date <= pd.Timestamp(cutoff_date_str).date() for last_date in vix_dict["last_date_dict"].values())
# DEFINE THE EXPLORATION WINDOW AND ONE PERIOD PER CALENDAR YEAR (FOR THE YEARS-WON COUNT)
exploration_start_str, exploration_end_str = exp_config.EXPLORATION_REPLAY_START_DATE_STR, exp_config.EXPLORATION_REPLAY_END_DATE_STR
session_date_arr = np.array(ohlcv_array_dict["session_date_list"])
window_date_arr = session_date_arr[(session_date_arr >= pd.Timestamp(exploration_start_str).date()) & (session_date_arr <= pd.Timestamp(exploration_end_str).date())]
if CHECK_REAL_DATA_BOOL:
    assert str(window_date_arr[0]) == exploration_start_str and str(window_date_arr[-1]) == exploration_end_str
year_period_pdf = pd.DataFrame([{"period_id": year, "period_start": min(d for d in window_date_arr if d.year == year), "period_end": max(d for d in window_date_arr if d.year == year)}
                                for year in sorted({d.year for d in window_date_arr})])
# DISPLAY THE LAST DATES LOADED AND THE WINDOW
print(f"Last SPY date loaded:\t{daily_pdf['date'].max()}\t({len(daily_pdf):,} sessions from {daily_pdf['date'].min()})")
for key_str, last_date in vix_dict["last_date_dict"].items():
    print(f"Last date loaded:\t{key_str}\t{last_date}")
print(f"Replay:\t{window_date_arr[0]} -> {window_date_arr[-1]}\t({len(window_date_arr):,} sessions, {len(year_period_pdf)} years)")
# DISPLAY THE COVERAGE OF THE THREE FEATURES IN THE WINDOW
window_feature_pdf = vix_feature_pdf[vix_feature_pdf["date"].isin(set(window_date_arr))]
print("Missing values in the window:", {col: int(window_feature_pdf[col].isna().sum()) for col in [exp_config.VIX_LEVEL_COL_STR, exp_config.VIX_TERM_COL_STR, exp_config.VIX_PCT250_COL_STR]})
print(f"Share of window decisions with vix_term_prev >= 1:\t{(window_feature_pdf[exp_config.VIX_TERM_COL_STR] >= 1).mean():.3f}"
      f"\twith vix_pct250_prev >= 0.9:\t{(window_feature_pdf[exp_config.VIX_PCT250_COL_STR] >= 0.9).mean():.3f}")

In [ ]:
"""
The 6 candidates and the 2 unmodified exits, each one continuous path (start invested, no forced buy-back), against
buy-and-hold over the same sessions.
"""
# SIMULATE BUY AND HOLD
buy_hold_dict = get_buy_and_hold_result_dict(ohlcv_array_dict, exploration_start_str, exploration_end_str)
print(f"Buy & hold:\ttotal return {buy_hold_dict['metric_dict']['total_return']:.2%} | Sharpe {buy_hold_dict['metric_dict']['sharpe_ratio']:.2f} | max drawdown {buy_hold_dict['metric_dict']['max_drawdown']:.2%}")
# DEFINE THE RULE BUILDER
rule_builder_func = get_rule_builder_func(daily_pdf, vix_feature_pdf)
# LISTS TO HOLD THE RESULTS
row_dict_list, scorecard_pdf_list = [], []
# ITERATE OVER THE UNMODIFIED EXITS (BASELINES, NOT LOGGED) AND THE 6 CANDIDATES
for candidate_dict in get_unmodified_candidate_list() + get_rule_candidate_list():
    # BUILD AND REPLAY THE CANDIDATE
    exit_signal_arr, reentry_func, reason_str = rule_builder_func(candidate_dict)
    simulation_dict = simulate_continuous_replay_dict(ohlcv_array_dict, exploration_start_str, exploration_end_str, exp_config.UNTOUCHED_START_DATE_STR,
                                                      volatility_arr_in=np.full(len(daily_pdf), np.nan), stop_k_in=None, reentry_func_in=reentry_func,
                                                      reentry_reason_str_in=reason_str, exit_signal_arr_in=exit_signal_arr, max_cash_sessions_in=exp_config.MAX_CASH_SESSIONS)
    # SUMMARIZE THE PATH (YEARS AS PERIODS) AND BUILD ITS SCORECARD
    summary_dict = get_replay_summary_dict(simulation_dict["daily_equity_pdf"], buy_hold_dict["daily_equity_pdf"], year_period_pdf, exp_config.BOOTSTRAP_BLOCK_MONTH_COUNT)
    scorecard_pdf = get_vix_scorecard_pdf(get_episode_scorecard_pdf(simulation_dict, ohlcv_array_dict), simulation_dict, vix_feature_pdf)
    scorecard_pdf_list.append(scorecard_pdf.assign(**candidate_dict))
    # STORE THE ROW
    trigger_series = scorecard_pdf["reentry_trigger"] if len(scorecard_pdf) else pd.Series(dtype=str)
    row_dict_list.append({**candidate_dict, "total_return": summary_dict["strategy_total_return"], "buy_hold_total_return": summary_dict["buy_hold_total_return"],
                          "excess_total_return": summary_dict["excess_total_return"], "annualized_return": summary_dict["strategy_annualized_return"],
                          "sharpe_ratio": summary_dict["strategy_sharpe_ratio"], "max_drawdown": summary_dict["strategy_max_drawdown"], "in_market_pct": summary_dict["in_market_pct"],
                          "exit_count": simulation_dict["metric_dict"]["signal_exit_count"], "mean_cash_sessions": simulation_dict["metric_dict"]["mean_cash_sessions"],
                          "bought_back_lower_pct": simulation_dict["metric_dict"]["positive_episode_pct"],
                          "product_s_over_r": float(np.prod(scorecard_pdf["s_over_r"])) if len(scorecard_pdf) else 1.0,
                          "vix_reentry_count": int(trigger_series.isin(["M1", "M2"]).sum()), "original_reentry_count": int((trigger_series == "original rule").sum()),
                          "years_won": summary_dict["period_windows_beating_buy_hold"], "year_count": summary_dict["period_window_count"],
                          "annualized_log_excess": summary_dict["log_excess_annualized_log_excess"], "log_excess_ci_low": summary_dict["log_excess_ci_low"], "log_excess_ci_high": summary_dict["log_excess_ci_high"]})
# CONVERT THE RESULTS AND ADD THE COMPARISON WITH THE UNMODIFIED EXIT OF EACH CANDIDATE
all_row_pdf = pd.DataFrame(row_dict_list)
unmodified_total_dict = all_row_pdf[all_row_pdf["modification"] == exp_config.UNMODIFIED_STR].set_index("exit_rule")["total_return"].to_dict()
all_row_pdf["unmodified_total_return"] = all_row_pdf["exit_rule"].map(unmodified_total_dict)
all_row_pdf["excess_vs_unmodified"] = all_row_pdf["total_return"] - all_row_pdf["unmodified_total_return"]
exploration_pdf = all_row_pdf[all_row_pdf["modification"] != exp_config.UNMODIFIED_STR].reset_index(drop=True)
exploration_scorecard_pdf = pd.concat(scorecard_pdf_list, ignore_index=True)
# PREVIEW DATAFRAME
all_row_pdf[["candidate_name", "total_return", "buy_hold_total_return", "excess_total_return", "unmodified_total_return", "excess_vs_unmodified", "in_market_pct",
             "exit_count", "vix_reentry_count", "original_reentry_count", "product_s_over_r", "max_drawdown", "annualized_log_excess", "log_excess_ci_low", "log_excess_ci_high"]].round(4)

In [ ]:
"""
Episode scorecard of every path. S / R > 1 = bought back lower than sold; reentry_trigger = M1, M2, original rule or end
of data; vix_at_exit / vix_at_reentry = vix_level_prev at the two decisions.
"""
# DISPLAY THE SCORECARDS
scorecard_col_list = ["exit_date", "exit_fill_price", "reentry_date", "reentry_fill_price", "s_over_r", "sessions_out", "reentry_trigger", "vix_at_exit", "vix_at_reentry",
                      "buy_hold_return_out", "max_decline_out_pct", "open_at_end"]
for candidate_name in all_row_pdf["candidate_name"]:
    candidate_scorecard_pdf = exploration_scorecard_pdf[exploration_scorecard_pdf["candidate_name"] == candidate_name] if len(exploration_scorecard_pdf) else exploration_scorecard_pdf
    print(f"\n{candidate_name}:\t{len(candidate_scorecard_pdf)} episodes, {int((candidate_scorecard_pdf['s_over_r'] > 1).sum()) if len(candidate_scorecard_pdf) else 0} bought back lower")
    if len(candidate_scorecard_pdf):
        display(candidate_scorecard_pdf[scorecard_col_list].round(4))

In [ ]:
# SAVE THE RESULTS
write_csv_file_to_path(all_row_pdf, f"{output_path_str}exploration_candidate_data.csv", "W")
write_csv_file_to_path(exploration_scorecard_pdf, f"{output_path_str}exploration_scorecard_data.csv", "W")

In [ ]:
# CHECK THE TRIAL BUDGET BEFORE LOGGING (EVERY STAGE OF THIS EXPERIMENT)
trial_log_pdf = read_trial_log_pdf()
experiment_trial_count = int((trial_log_pdf["experiment"] == exp_config.EXPERIMENT_NAME).sum()) if len(trial_log_pdf) else 0
assert experiment_trial_count + len(exploration_pdf) <= exp_config.TRIAL_BUDGET, "❌ Trial budget exceeded"
# LOG THE 6 CANDIDATES AS EXPLORATION TRIALS (ONE WRITE)
exploration_trial_pdf = log_trial_pdf(exp_config, "exploration", exploration_pdf.assign(window=f"{exploration_start_str}->{exploration_end_str}"), ["exit_rule", "modification", "window"],
                                      [col for col in exploration_pdf.columns if col not in ["exit_rule", "modification", "candidate_name", "tie_order"]], False,
                                      note_str_in="exp12 step01 exploration (2009-08-13 -> 2014-12-31 continuous replay, context only)")
print(f"Trials logged:\t{len(exploration_trial_pdf)}\tfirst logged_ts {exploration_trial_pdf['logged_ts'].iloc[0]}\ttest_window_touched: {exploration_trial_pdf['test_window_touched'].unique().tolist()}")

In [ ]:
# CALL FUNCTION TO READ THE TRIAL LOG
trial_log_pdf = read_trial_log_pdf()
# DISPLAY THE NUMBER OF TRIALS PER EXPERIMENT AND STAGE (MULTIPLE-TESTING RECORD)
get_trial_count_pdf(trial_log_pdf)